In [2]:
import pandas as pd
import json
import os
from pathlib import Path


In [3]:
# CSV já consolidado e avaliado pela rubrica de 16 dimensões
CAMINHO_CSV = Path("..") / "resultado_rubrica" / "prova-tecnologia-results-1500-rubrica-16dim.csv"

# Mapeia prompt_variant → nome exibido nas tabelas
ROTULO_PROMPT = {
    "base": "Base",
    "nicol": "Nicol",
    "hattie": "Hattie",
}

In [4]:
def extrair_average_score(valor):
    """Média das 16 dimensões normalizada para escala 0–2."""
    try:
        if pd.isna(valor):
            return None

        dados = json.loads(valor) if isinstance(valor, str) else valor

        if "erro" in dados:
            return None

        # formato antigo (biologia / chatgpt)
        if "overall" in dados:
            return dados["overall"].get("average_score")
        if "evaluation" in dados:
            return dados["evaluation"]["overall"].get("average_score")

        # formato 16 dimensões (resultado_rubrica)
        scores = []
        for info in dados.get("conteudo", {}).values():
            scores.append(info["score"])  # 0–2
        for info in dados.get("efetividade", {}).values():
            scores.append(info["score"] * 2)  # 0–1 → 0–2
        for info in dados.get("alucinacoes", {}).values():
            scores.append((1 - info["score"]) * 2)  # 0=sem alucinação → 2

        return sum(scores) / len(scores) if scores else None

    except Exception:
        return None

In [5]:
if not CAMINHO_CSV.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {CAMINHO_CSV.resolve()}")

base_consolidada = pd.read_csv(CAMINHO_CSV, sep=None, engine="python")
base_consolidada.columns = base_consolidada.columns.str.strip()

if "analise_feedback" not in base_consolidada.columns:
    raise ValueError(
        "Coluna 'analise_feedback' não encontrada. "
        f"Colunas disponíveis: {base_consolidada.columns.tolist()}"
    )

# modelo e prompt_variant já vêm do CSV consolidado
base_consolidada["rubrica"] = (
    base_consolidada["prompt_variant"].astype(str).str.lower().map(ROTULO_PROMPT)
)
base_consolidada["average_score"] = base_consolidada["analise_feedback"].apply(extrair_average_score)

print(f"Linhas: {len(base_consolidada)}")
print(f"Scores válidos: {base_consolidada['average_score'].notna().sum()}")
print(f"Sem score (erro/JSON inválido): {base_consolidada['average_score'].isna().sum()}")

base_consolidada[["modelo", "rubrica", "average_score"]].head()

Linhas: 1500
Scores válidos: 1249
Sem score (erro/JSON inválido): 251


,modelo,rubrica,average_score
0,phi4-mini,Hattie,1.4375
1,qwen3,Base,1.5000
2,phi4-mini,Hattie,1.4375
3,qwen3,Base,1.5000
4,gemma4,Hattie,1.1250


In [6]:
tabela_modelo_rubrica = (
    base_consolidada
    .dropna(subset=["average_score"])
    .groupby(["modelo", "rubrica"], as_index=False)
    .agg(
        average_score_medio=("average_score", "mean"),
        qtd_avaliacoes=("average_score", "count"),
    )
    .sort_values(["modelo", "rubrica"])
)

tabela_modelo_rubrica

,modelo,rubrica,average_score_medio,qtd_avaliacoes
0,gemma4,Base,1.434160,131
1,gemma4,Hattie,1.448902,148
2,gemma4,Nicol,1.488971,136
3,phi4-mini,Base,1.399194,124
4,phi4-mini,Hattie,1.316993,153
5,phi4-mini,Nicol,1.391304,138
6,qwen3,Base,1.361168,122
7,qwen3,Hattie,1.328125,152
8,qwen3,Nicol,1.451724,145


In [7]:
tabela_modelo_rubrica_pivot = (
    tabela_modelo_rubrica
    .pivot(
        index="modelo",
        columns="rubrica",
        values="average_score_medio"
    )
    .reset_index()
)

tabela_modelo_rubrica_pivot

rubrica,modelo,Base,Hattie,Nicol
0,gemma4,1.434160,1.448902,1.488971
1,phi4-mini,1.399194,1.316993,1.391304
2,qwen3,1.361168,1.328125,1.451724
